# Denoising exploratory analysis

Moving-average and median-filter smoothers on a generation series. This notebook is exploratory: window sizes were chosen by inspection, not by a held-out criterion.

Uses `data/synthetic/sample_site_0.csv` so a public clone runs without proprietary site data. If you have local real data at `data/real/validation_Data.csv`, that file is preferred.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage

ROOT = Path("..")
real_path = ROOT / "data" / "real" / "validation_Data.csv"
synth_path = ROOT / "data" / "synthetic" / "sample_site_0.csv"

if real_path.exists():
    df = pd.read_csv(real_path)
    site_id = df["site_id"].iloc[0]
    df = df.loc[df["site_id"] == site_id].copy()
    source = f"real site {site_id} (local, not in public repo)"
else:
    df = pd.read_csv(synth_path)
    source = "synthetic sample_site_0"

df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
data = df["solar_gen_kwh"].fillna(0).to_numpy()
time = df["timestamp_utc"]
print(source, "n =", len(df))

In [ ]:
window_size = 50
smoothed_ma = np.convolve(data, np.ones(window_size) / window_size, mode="same")
smoothed_med = ndimage.median_filter(data, size=window_size, mode="reflect")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(time, data, ".", color="tab:red", markersize=1, alpha=0.4, label="raw")
ax.plot(time, smoothed_ma, color="tab:blue", linewidth=1, label=f"moving average (w={window_size})")
ax.plot(time, smoothed_med, color="tab:green", linewidth=1, label=f"median filter (w={window_size})")
ax.set_title(f"Generation smoothers — {source}")
ax.set_xlabel("UTC")
ax.set_ylabel("solar_gen_kwh")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
removed_zeros = data[data != 0]
smoothed_nonzero = ndimage.median_filter(removed_zeros, size=window_size, mode="reflect")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(removed_zeros, ".", markersize=0.5, alpha=0.5, label="nonzero raw")
ax.plot(smoothed_nonzero, color="tab:green", linewidth=1, label="median filter")
ax.set_title("Same filters after dropping exact zeros")
ax.set_xlabel("sample index")
ax.set_ylabel("solar_gen_kwh")
ax.legend()
plt.tight_layout()
plt.show()